# CIC-IDS2017 - E3 controlled imbalance sweep (BENIGN vs Bot)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/cicids2017/e3_ir_sweep/cicids2017_e3_ir_sweep.ipynb)

Only the imbalance ratio changes: N = 10,000 flows at minority share [0.15, 0.1, 0.05, 0.02, 0.01].
All 9 configs are run with LR, RF, XGB and 10 search candidates
per fold (reduced budget because this is 5 levels x 9 configs).

To split the work across Colab sessions, set `LEVELS` to a subset, e.g. `[0.15, 0.10]` in one
session and `[0.05, 0.02, 0.01]` in another. Both append to the same file.

**How to run on Colab**
1. Click the badge, then *Runtime > Run all*. Allow Google Drive access when asked.
2. Results are appended to Google Drive after every classifier/fold:
   `MyDrive/ML-Research-Dataset-Imbalance/cicids2017/e3_ir_sweep/results/cicids2017__e3_ir_sweep.csv`.
3. If the runtime disconnects, just *Run all* again: finished folds are skipped.
4. When it is complete, copy the CSV into the same path in the repository and commit it.

In [ ]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


In [ ]:
LEVELS = None   # None = all levels
# Quick test run (2 folds, fewer classifiers and candidates): change the next line
# to  SMOKE = True . Its results go to a separate *__SMOKE.csv file and must never
# be used in the paper. Leave it as it is for the real run.
SMOKE = os.environ.get("SMOKE", "0") == "1"

In [ ]:
from common.cv import run_e3
results = run_e3(LEVELS, smoke=SMOKE)

## Macro-F1 by imbalance level and config (mean over classifiers and folds)

In [ ]:
results.pivot_table(index="config_id", columns="ir_level", values="macro_f1").round(4)